In [0]:
# ============================================================
# CELL 1: Imports & Read Bronze Layer
# ============================================================

from pyspark.sql.functions import (
    col, trim, upper, regexp_replace, when, lit,
    to_date, current_timestamp, sum as spark_sum
)
from pyspark.sql.types import IntegerType

bronze_path = "/Volumes/healthcare_pipeline/ae_data/bronze_volume/delta/ae_attendances"

bronze_df = spark.read.format("delta").load(bronze_path)

print(f"Records read from Bronze: {bronze_df.count()}")
print(f"Columns: {len(bronze_df.columns)}")
print()
print("Column list:")
for c in bronze_df.columns:
    print(f"  {c}")

Records read from Bronze: 2382
Columns: 25

Column list:
  period
  org_code
  parent_org
  org_name
  a_e_attendances_type_1
  a_e_attendances_type_2
  a_e_attendances_other_a_e_department
  a_e_attendances_booked_appointments_type_1
  a_e_attendances_booked_appointments_type_2
  a_e_attendances_booked_appointments_other_department
  attendances_over_4hrs_type_1
  attendances_over_4hrs_type_2
  attendances_over_4hrs_other_department
  attendances_over_4hrs_booked_appointments_type_1
  attendances_over_4hrs_booked_appointments_type_2
  attendances_over_4hrs_booked_appointments_other_department
  patients_who_have_waited_4_12_hs_from_dta_to_admission
  patients_who_have_waited_12_hrs_from_dta_to_admission
  emergency_admissions_via_a_e_type_1
  emergency_admissions_via_a_e_type_2
  emergency_admissions_via_a_e_other_a_e_department
  other_emergency_admissions
  ingestion_timestamp
  source_file
  pipeline_version


In [0]:
# ============================================================
# CELL 2: Inspect Period Column Format
# ============================================================

bronze_df.select("period").distinct().orderBy("period").show(20, truncate=False)

+---------------------+
|period               |
+---------------------+
|MSitAE-APRIL-2026    |
|MSitAE-AUGUST-2025   |
|MSitAE-DECEMBER-2025 |
|MSitAE-FEBRUARY-2026 |
|MSitAE-JANUARY-2026  |
|MSitAE-JULY-2025     |
|MSitAE-JUNE-2025     |
|MSitAE-MARCH-2026    |
|MSitAE-MAY-2026      |
|MSitAE-NOVEMBER-2025 |
|MSitAE-OCTOBER-2025  |
|MSitAE-SEPTEMBER-2025|
|TOTAL                |
|TOTAL                |
+---------------------+



In [0]:
# ============================================================
# CELL 3: Clean Numeric Columns
# ============================================================

def clean_numeric_col(df, col_name):
    """
    Cleans a numeric column stored as string:
    - Removes commas (e.g. '1,234' -> '1234')
    - Replaces '-' (NHS null indicator) with 0
    - Casts to Integer
    - Fills any remaining nulls with 0
    """
    return df.withColumn(
        col_name,
        when(col(col_name) == "-", lit(0))
        .otherwise(regexp_replace(col(col_name), ",", ""))
        .cast(IntegerType())
    ).fillna({col_name: 0})

# List of all numeric columns to clean
numeric_columns = [
    "a_e_attendances_type_1",
    "a_e_attendances_type_2",
    "a_e_attendances_other_a_e_department",
    "a_e_attendances_booked_appointments_type_1",
    "a_e_attendances_booked_appointments_type_2",
    "a_e_attendances_booked_appointments_other_department",
    "attendances_over_4hrs_type_1",
    "attendances_over_4hrs_type_2",
    "attendances_over_4hrs_other_department",
    "attendances_over_4hrs_booked_appointments_type_1",
    "attendances_over_4hrs_booked_appointments_type_2",
    "attendances_over_4hrs_booked_appointments_other_department",
    "patients_who_have_waited_4_12_hs_from_dta_to_admission",
    "patients_who_have_waited_12_hrs_from_dta_to_admission",
    "emergency_admissions_via_a_e_type_1",
    "emergency_admissions_via_a_e_type_2",
    "emergency_admissions_via_a_e_other_a_e_department",
    "other_emergency_admissions",
]

silver_df = bronze_df

for c in numeric_columns:
    silver_df = clean_numeric_col(silver_df, c)

print(f"Cleaned {len(numeric_columns)} numeric columns")
print()
silver_df.select(numeric_columns[:4]).show(5)

Cleaned 18 numeric columns

+----------------------+----------------------+------------------------------------+------------------------------------------+
|a_e_attendances_type_1|a_e_attendances_type_2|a_e_attendances_other_a_e_department|a_e_attendances_booked_appointments_type_1|
+----------------------+----------------------+------------------------------------+------------------------------------------+
|                     0|                     0|                                   0|                                         0|
|                     0|                     0|                                3100|                                         0|
|                     0|                     0|                                6125|                                         0|
|                     0|                     0|                                1793|                                         0|
|                     0|                     0|                             

In [0]:
# ============================================================
# CELL 4 (Fixed): Parse Period to Date & Remove TOTAL Rows
# ============================================================

from pyspark.sql.functions import regexp_extract, concat_ws, to_date

# Step 1: Remove summary/TOTAL rows
print(f"Records before removing TOTAL rows: {silver_df.count()}")

silver_df = silver_df.filter(col("period") != "TOTAL")
silver_df = silver_df.filter(~col("org_code").isin(["TOTAL", "-"]))

print(f"Records after removing TOTAL rows: {silver_df.count()}")
print()

# Step 2: Check for any period values that DON'T match expected pattern
bad_rows = silver_df.filter(~col("period").rlike(r"^MSitAE-[A-Z]+-\d{4}$"))
print(f"Rows with unexpected period format: {bad_rows.count()}")
if bad_rows.count() > 0:
    bad_rows.select("period", "org_code").distinct().show(20, truncate=False)

# Step 3: Remove any rows that don't match the expected period pattern
silver_df = silver_df.filter(col("period").rlike(r"^MSitAE-[A-Z]+-\d{4}$"))
print(f"Records after removing malformed period rows: {silver_df.count()}")
print()

# Step 4: Extract month name and year
silver_df = silver_df \
    .withColumn("_month_name", regexp_extract(col("period"), r"MSitAE-([A-Z]+)-(\d{4})", 1)) \
    .withColumn("_year", regexp_extract(col("period"), r"MSitAE-([A-Z]+)-(\d{4})", 2))

# Step 5: Map month name to month number
silver_df = silver_df.withColumn(
    "_month_num",
    when(col("_month_name") == "JANUARY", "01")
    .when(col("_month_name") == "FEBRUARY", "02")
    .when(col("_month_name") == "MARCH", "03")
    .when(col("_month_name") == "APRIL", "04")
    .when(col("_month_name") == "MAY", "05")
    .when(col("_month_name") == "JUNE", "06")
    .when(col("_month_name") == "JULY", "07")
    .when(col("_month_name") == "AUGUST", "08")
    .when(col("_month_name") == "SEPTEMBER", "09")
    .when(col("_month_name") == "OCTOBER", "10")
    .when(col("_month_name") == "NOVEMBER", "11")
    .when(col("_month_name") == "DECEMBER", "12")
)

# Step 6: Build proper date column
silver_df = silver_df.withColumn(
    "period_date",
    to_date(concat_ws("-", col("_year"), col("_month_num"), lit("01")), "yyyy-MM-dd")
)

# Step 7: Drop helper columns
silver_df = silver_df.drop("_month_name", "_year", "_month_num")

# Verify
print("Parsed dates - sample:")
silver_df.select("period", "period_date").distinct().orderBy("period_date").show(15, truncate=False)

# Final null check on the new date column
null_dates = silver_df.filter(col("period_date").isNull()).count()
print(f"\nRows with NULL period_date after parsing: {null_dates}")

Records before removing TOTAL rows: 2371
Records after removing TOTAL rows: 2371

Rows with unexpected period format: 1
+------+--------+
|period|org_code|
+------+--------+
|TOTAL |TOTAL   |
+------+--------+

Records after removing malformed period rows: 2370

Parsed dates - sample:
+---------------------+-----------+
|period               |period_date|
+---------------------+-----------+
|MSitAE-JUNE-2025     |2025-06-01 |
|MSitAE-JULY-2025     |2025-07-01 |
|MSitAE-AUGUST-2025   |2025-08-01 |
|MSitAE-SEPTEMBER-2025|2025-09-01 |
|MSitAE-OCTOBER-2025  |2025-10-01 |
|MSitAE-NOVEMBER-2025 |2025-11-01 |
|MSitAE-DECEMBER-2025 |2025-12-01 |
|MSitAE-JANUARY-2026  |2026-01-01 |
|MSitAE-FEBRUARY-2026 |2026-02-01 |
|MSitAE-MARCH-2026    |2026-03-01 |
|MSitAE-APRIL-2026    |2026-04-01 |
|MSitAE-MAY-2026      |2026-05-01 |
+---------------------+-----------+


Rows with NULL period_date after parsing: 0


In [0]:
# ============================================================
# CELL 5: Remove Duplicates & Clean Text Columns
# ============================================================

# Standardise text columns
silver_df = silver_df \
    .withColumn("org_code", upper(trim(col("org_code")))) \
    .withColumn("org_name", trim(col("org_name"))) \
    .withColumn("parent_org", trim(col("parent_org")))

# Check for duplicates before removing
total_before = silver_df.count()

# A record is duplicate if same period + org_code appears more than once
silver_df = silver_df.dropDuplicates(["period_date", "org_code"])

total_after = silver_df.count()
duplicates_removed = total_before - total_after

print(f"Records before dedup: {total_before}")
print(f"Records after dedup:  {total_after}")
print(f"Duplicates removed:   {duplicates_removed}")
print()

# Add Silver metadata timestamp
silver_df = silver_df.withColumn("silver_processed_timestamp", current_timestamp())

print("Silver cleaning complete")

Records before dedup: 2370
Records after dedup:  2370
Duplicates removed:   0

Silver cleaning complete


In [0]:
# ============================================================
# CELL 6: Data Quality Checks
# ============================================================

print("=== SILVER LAYER DATA QUALITY REPORT ===")
print()

# Check 1: Null check on critical columns
critical_cols = ["period_date", "org_code", "org_name"]
for c in critical_cols:
    null_count = silver_df.filter(col(c).isNull()).count()
    status = "PASS" if null_count == 0 else "FAIL"
    print(f"[{status}] Null check on '{c}': {null_count} nulls found")

print()

# Check 2: Volume check - should have 12 distinct months
month_count = silver_df.select("period_date").distinct().count()
status = "PASS" if month_count == 12 else "WARNING"
print(f"[{status}] Distinct months in data: {month_count} (expected 12)")

print()

# Check 3: Business rule - 4hr breaches cannot exceed total attendances
total_attendances_expr = (
    col("a_e_attendances_type_1") + col("a_e_attendances_type_2") +
    col("a_e_attendances_other_a_e_department")
)
total_breaches_expr = (
    col("attendances_over_4hrs_type_1") + col("attendances_over_4hrs_type_2") +
    col("attendances_over_4hrs_other_department")
)

invalid_breaches = silver_df.filter(total_breaches_expr > total_attendances_expr).count()
status = "PASS" if invalid_breaches == 0 else "FAIL"
print(f"[{status}] Breach rate sanity check: {invalid_breaches} invalid records (breaches > attendances)")

print()

# Check 4: Negative value check
negative_check = silver_df.filter(col("a_e_attendances_type_1") < 0).count()
status = "PASS" if negative_check == 0 else "FAIL"
print(f"[{status}] Negative values check: {negative_check} negative attendance records")

print()

# Check 5: Distinct hospitals
hospital_count = silver_df.select("org_code").distinct().count()
print(f"[INFO] Distinct hospitals/organisations in data: {hospital_count}")

print()

# Check 6: Total volume summary
total_attendances = silver_df.agg(spark_sum("a_e_attendances_type_1")).collect()[0][0]
print(f"[INFO] Total Type 1 A&E attendances across all periods: {total_attendances:,}")

=== SILVER LAYER DATA QUALITY REPORT ===

[PASS] Null check on 'period_date': 0 nulls found
[PASS] Null check on 'org_code': 0 nulls found
[PASS] Null check on 'org_name': 0 nulls found

[PASS] Distinct months in data: 12 (expected 12)

[PASS] Breach rate sanity check: 0 invalid records (breaches > attendances)

[PASS] Negative values check: 0 negative attendance records

[INFO] Distinct hospitals/organisations in data: 205

[INFO] Total Type 1 A&E attendances across all periods: 16,841,461


In [0]:
spark.sql("USE CATALOG healthcare_pipeline")
spark.sql("USE DATABASE ae_data")
spark.sql("CREATE VOLUME IF NOT EXISTS silver_volume")
print("Silver volume created")

Silver volume created


In [0]:
# ============================================================
# CELL 7: Write Silver Delta Table
# ============================================================

silver_path = "/Volumes/healthcare_pipeline/ae_data/silver_volume/delta/ae_attendances_clean"

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_path)

print("Silver Delta table written successfully")
print()

# Verify
verify_silver = spark.read.format("delta").load(silver_path)
print(f"Records in Silver Delta table: {verify_silver.count()}")
print(f"Columns in Silver Delta table: {len(verify_silver.columns)}")
print()
print("Silver layer complete!")

Silver Delta table written successfully

Records in Silver Delta table: 2370
Columns in Silver Delta table: 27

Silver layer complete!
